In [91]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('engine_failure_dataset.csv')
df.head(10)

,Time_Stamp,Temperature,RPM,Fuel_Efficiency,Vibration_X,Vibration_Y,Vibration_Z,Torque,Power_Output,Fault_Condition,Operational_Mode
0,2024-12-24 10:00:00,60.308585,3426.827588,20.445472,0.874657,0.005686,0.529798,107.877658,23.367684,2,Idle
1,2024-12-24 10:05:00,112.705055,2949.758424,23.083947,0.696461,0.391779,0.124336,60.351655,57.941022,3,Cruising
2,2024-12-24 10:10:00,108.670976,1817.971040,20.555326,0.495276,0.189714,0.886417,110.986564,47.732998,2,Cruising
3,2024-12-24 10:15:00,107.114691,2730.660539,23.226431,0.986206,0.983202,0.468114,77.416793,44.112039,2,Cruising
4,2024-12-24 10:20:00,118.075814,1854.488677,21.148226,0.710810,0.101139,0.481034,100.475881,80.681972,2,Cruising
5,2024-12-24 10:25:00,78.157210,2324.480476,16.627448,0.484482,0.242013,0.437504,91.495077,86.981820,0,Cruising
6,2024-12-24 10:30:00,114.084534,1128.439288,26.561841,0.261974,0.715318,0.280295,123.156877,82.441668,1,Cruising
7,2024-12-24 10:35:00,108.399936,2424.950038,24.155412,0.432189,0.761122,0.869901,152.349026,80.371809,1,Idle
8,2024-12-24 10:40:00,84.485655,1662.500402,25.535557,0.140342,0.341326,0.172818,90.763384,27.128899,0,Idle
9,2024-12-24 10:45:00,73.573443,2815.996636,28.375972,0.384356,0.034136,0.230663,127.348256,31.823300,1,Heavy Load


In [92]:
df = df.drop('Time_Stamp', axis=1)

In [93]:
# Get dimensions
rows = df.shape[0]
attributes = df.shape[1]

print(f"Number of rows: {rows}")
print(f"Number of attributes: {attributes}")
df.shape

Number of rows: 1000
Number of attributes: 10


(1000, 10)

In [94]:
numeric_count = len(df.select_dtypes(include=['number']).columns)

# Count categorical attributes (objects, strings, categories)
categorical_count = len(df.select_dtypes(exclude=['number']).columns)

print(f"Numeric attributes: {numeric_count}")
print(f"Categorical attributes: {categorical_count}")

Numeric attributes: 9
Categorical attributes: 1


In [95]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Temperature       1000 non-null   float64
 1   RPM               1000 non-null   float64
 2   Fuel_Efficiency   1000 non-null   float64
 3   Vibration_X       1000 non-null   float64
 4   Vibration_Y       1000 non-null   float64
 5   Vibration_Z       1000 non-null   float64
 6   Torque            1000 non-null   float64
 7   Power_Output      1000 non-null   float64
 8   Fault_Condition   1000 non-null   int64  
 9   Operational_Mode  1000 non-null   object 
dtypes: float64(8), int64(1), object(1)
memory usage: 74.3+ KB


In [102]:
import micropip

await micropip.install("seaborn")

In [119]:
numeric_df = df.select_dtypes(include=['int64', 'float64'])
print(numeric_df)

     Temperature          RPM  Fuel_Efficiency  Vibration_X  Vibration_Y  \
0      60.308585  3426.827588        20.445472     0.874657     0.005686   
1     112.705055  2949.758424        23.083947     0.696461     0.391779   
2     108.670976  1817.971040        20.555326     0.495276     0.189714   
3     107.114691  2730.660539        23.226431     0.986206     0.983202   
4     118.075814  1854.488677        21.148226     0.710810     0.101139   
..           ...          ...              ...          ...          ...   
995    88.231211  1477.774501        23.749532     0.827987     0.664745   
996   105.241946  1617.745044        16.166110     0.753548     0.797969   
997   119.066775  2473.669785        17.657404     0.982960     0.699409   
998    90.620157  2297.744136        17.479882     0.686213     0.875040   
999    93.144406  2511.308863        17.597145     0.950341     0.340887   

     Vibration_Z      Torque  Power_Output  Fault_Condition  
0       0.529798  107.877

In [120]:
numeric_df = numeric_df.drop('Fault_Condition', axis=1)

In [122]:
# Standardize data
scaler = StandardScaler()

scaled_data = scaler.fit_transform(numeric_df)

In [139]:
# Separate features and target
X = df.drop('Fault_Condition', axis=1)
y = df['Fault_Condition']

# Keep only numeric features
X = X.select_dtypes(include=['int64', 'float64'])

# Handle missing values
X = X.fillna(X.median())

# Normalize data
#scaler = StandardScaler()

#X_scaled = scaler.fit_transform(X)

# Apply KBest
selector = SelectKBest(
    score_func=chi2,
    k=3
)

X_new = selector.fit_transform(X, y)

# Selected features
selected_features = X.columns[selector.get_support()]

print("Selected Features:")
print(selected_features)

# Feature scores
scores = pd.DataFrame({
    'Feature': X.columns,
    'Chi2 Score': selector.scores_
})

print(scores.sort_values(
    by='Chi2 Score',
    ascending=False
))


Selected Features:
Index(['RPM', 'Torque', 'Power_Output'], dtype='object')
           Feature  Chi2 Score
6           Torque  131.553024
1              RPM   91.244165
7     Power_Output   22.697476
2  Fuel_Efficiency    4.328522
0      Temperature    3.707427
3      Vibration_X    0.593974
4      Vibration_Y    0.345148
5      Vibration_Z    0.147264
